# 🇵🇰 Pakistani Social Media Content Detective
### Phone-brand perception pipeline: comments → sentiment → problems → trust → recommendation

**How to use (about 2–3 hours total):**
1. **Run cell 1** and edit the config: your `BRAND`, and `SOURCE` (`"youtube"`, `"paste"`, `"csv"` or `"demo"`).
2. **Collect comments** (cell 2 does it for you): YouTube API, or paste comments you copied from *public* Facebook groups / Instagram / TikTok.
3. **Run all cells.** Cleaning, labeling, 10 charts, tests and an AI recommendation prompt are automatic.
4. **Validate** (cell 5): label 150 comments by hand once, upload, and get accuracy + kappa. Markers love this.
5. **Export** (last cell) gives you a zip: labeled data, tables, charts, and a report draft with your numbers filled in.

⚠️ `SOURCE = "demo"` uses **fake generated comments** so you can test the pipeline. Never submit demo results.

In [ ]:
# ===================== 1. SETUP & CONFIG (edit this cell only) =====================
import os, re, io, json, time, hashlib, random, warnings, subprocess, sys
from collections import Counter
warnings.filterwarnings("ignore")

try:
    import pandas, numpy, matplotlib, seaborn, scipy, sklearn, openpyxl, requests
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas", "numpy",
                           "matplotlib", "seaborn", "scipy", "scikit-learn", "openpyxl", "requests"])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from scipy import stats

try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ---- WHAT ARE YOU STUDYING? ----
BRAND = "Infinix"                 # the one phone brand you picked (used in prompts + stop-words)
MODEL_KEYWORDS = []               # optional, e.g. ["hot 40", "note 30", "spark 20"] -> adds a 'model' column

# ---- WHERE DO THE COMMENTS COME FROM? ----
SOURCE = "demo"                   # "youtube" | "paste" | "csv" | "demo"

# (a) YouTube: free API key from console.cloud.google.com -> enable "YouTube Data API v3"
YOUTUBE_API_KEY = ""
YOUTUBE_VIDEOS = [                # paste full links or 11-char IDs of Pakistani review/unboxing videos
    # "https://www.youtube.com/watch?v=XXXXXXXXXXX",
]
MAX_COMMENTS_PER_VIDEO = 500
YOUTUBE_ORDER = "relevance"       # "relevance" (top comments) or "time" (newest first)

# (b) Paste mode: copy comments from PUBLIC Facebook groups / Instagram / TikTok posts.
#     One comment per line. Optional likes: "12 | comment text". Add one dict per post.
PASTE_BATCHES = [
    # {"platform": "Facebook Group", "source": "Mobile Bazar Lahore - post 1",
    #  "post_type": "experience_review",       # experience_review | buying_question | sale_listing
    #  "date": "2026-10-01",                   # YYYY-MM-DD (add time like 2026-10-01 18:30 if you know it)
    #  "text": """
    #  Battery bohat acha hai
    #  5 | Phone garam hota hai, mat lo
    #  Which model is best under 40k?
    #  """},
]

# (c) CSV mode: columns platform, source, post_type, date, text, likes, reply_count  (only `text` is required)
CSV_PATH = "comments_template.csv"

# ---- HOW TO LABEL SENTIMENT / ASPECTS? ----
LABEL_MODE = "rules"              # "rules" = free + offline, "claude" = better on Roman Urdu (needs API key)
ANTHROPIC_API_KEY = ""            # only for LABEL_MODE="claude" and the AI recommendation
LLM_MODEL = "claude-haiku-5-5"    # cheap + fast. Use "claude-sonnet-5-5" for the recommendation if you like

# ---- VALIDATION ----
GOLD_LABELED_PATH = ""            # after hand-labeling (cell 5), set to "gold_labeled.csv" and re-run cell 5

OUT_DIR = "outputs"
os.makedirs(f"{OUT_DIR}/charts", exist_ok=True)

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams.update({"figure.figsize": (10, 5), "axes.titlesize": 13, "axes.titleweight": "bold"})
SENT_ORDER = ["Positive", "Neutral", "Negative"]
COLORS = {"Positive": "#43a047", "Neutral": "#9e9e9e", "Negative": "#e53935"}
DAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
TAG = "  [DEMO DATA]" if SOURCE == "demo" else ""
ASPECT_NAMES = ["Battery", "Camera", "Heating", "Performance", "Display", "Price/Value",
                "Software", "Build/Design", "After-sales/Warranty", "Authenticity/Trust"]
print(f"Brand: {BRAND} | Source: {SOURCE} | Labeling: {LABEL_MODE} | Colab: {IN_COLAB}")

In [ ]:
# ===================== 2. COLLECT COMMENTS =====================
REQUIRED_COLS = ["platform", "source", "post_type", "date", "text", "likes", "reply_count"]

def youtube_id(s):
    s = s.strip()
    m = re.search(r"(?:v=|youtu\.be/|shorts/|embed/)([A-Za-z0-9_-]{11})", s)
    if m:
        return m.group(1)
    return s if re.fullmatch(r"[A-Za-z0-9_-]{11}", s) else None

def fetch_youtube(videos, api_key, max_per_video, order):
    if not api_key:
        raise ValueError("Add YOUTUBE_API_KEY in cell 1 (free: console.cloud.google.com -> YouTube Data API v3).")
    rows = []
    for v in videos:
        vid = youtube_id(v)
        if not vid:
            print("  skipped (bad link):", v); continue
        title = vid
        r = requests.get("https://www.googleapis.com/youtube/v3/videos",
                         params={"part": "snippet", "id": vid, "key": api_key}, timeout=30)
        if r.ok and r.json().get("items"):
            title = r.json()["items"][0]["snippet"]["title"][:70]
        got, token = 0, None
        while got < max_per_video:
            params = {"part": "snippet", "videoId": vid, "maxResults": 100, "order": order,
                      "textFormat": "plainText", "key": api_key}
            if token:
                params["pageToken"] = token
            r = requests.get("https://www.googleapis.com/youtube/v3/commentThreads", params=params, timeout=30)
            if not r.ok:
                msg = r.json().get("error", {}).get("message", r.text[:150]) if r.headers.get("content-type", "").startswith("application/json") else r.text[:150]
                print(f"  {vid}: API error -> {msg}"); break
            data = r.json()
            for it in data.get("items", []):
                s = it["snippet"]["topLevelComment"]["snippet"]
                rows.append({"platform": "YouTube", "source": title, "post_type": "video",
                             "date": s.get("publishedAt"), "text": s.get("textDisplay", ""),
                             "likes": s.get("likeCount", 0), "reply_count": it["snippet"].get("totalReplyCount", 0)})
                got += 1
            token = data.get("nextPageToken")
            if not token:
                break
        print(f"  {title}: {got} comments")
    return pd.DataFrame(rows)

def load_paste(batches):
    rows = []
    for b in batches:
        for line in b["text"].splitlines():
            line = line.strip()
            if not line:
                continue
            likes = 0
            m = re.match(r"^(\d+)\s*\|\s*(.+)$", line)
            if m:
                likes, line = int(m.group(1)), m.group(2)
            rows.append({"platform": b.get("platform", "Facebook Group"), "source": b.get("source", ""),
                         "post_type": b.get("post_type", "unknown"), "date": b.get("date"),
                         "text": line, "likes": likes, "reply_count": 0})
    return pd.DataFrame(rows)

def load_csv(path):
    if not os.path.exists(path) and IN_COLAB:
        from google.colab import files
        print("Upload your comments CSV:")
        up = files.upload(); path = list(up.keys())[0]
    d = pd.read_csv(path)
    for c in REQUIRED_COLS:
        if c not in d.columns:
            d[c] = {"platform": "Unknown", "source": "", "post_type": "unknown", "date": None, "likes": 0, "reply_count": 0}.get(c)
    return d[REQUIRED_COLS]

def make_demo(n=360, seed=7):
    """FAKE comments to test the pipeline. Never use for your real findings."""
    rng = random.Random(seed)
    T = {
     ("Positive", "Battery"): ["Battery backup is amazing, lasts a full day", "battery bohat acha hai, 2 din chalti hai", "Charging is super fast, love it"],
     ("Negative", "Battery"): ["Battery drains too fast, very disappointed", "battery bakwas hai, dopahar tak khatam", "Charging is slow and battery is poor"],
     ("Positive", "Camera"): ["Camera quality is great for this price", "camera zabardast hai, selfie bohat achi aati hai"],
     ("Negative", "Camera"): ["Camera is bad in low light", "camera kharab hai, photos blur aati hain"],
     ("Negative", "Heating"): ["Phone heats a lot while gaming, big problem", "phone bohat garam hota hai, pubg mein hang bhi karta hai", "overheating issue after the update"],
     ("Positive", "Performance"): ["Very smooth performance, no lag", "phone fast hai aur smooth chalta hai"],
     ("Negative", "Performance"): ["Lag and hang every day, useless", "bohat slow hai, hang hota rehta hai"],
     ("Positive", "Price"): ["paisa wasool phone, best in this budget", "Great phone for the price"],
     ("Negative", "Price"): ["Too expensive for these specs", "mehnga hai is quality ke hisaab se, overpriced"],
     ("Negative", "Service"): ["Warranty claim was a nightmare, customer care is useless", "service center walay bekar hain, 2 hafte repair mein"],
     ("Negative", "Trust"): ["Got a refurbished piece from the shop, never again", "non pta set tha, dhoka hua. dont trust this brand", "brand pe bharosa nahi raha, scam"],
     ("Negative", "Software"): ["Too many ads in the UI, software is bad", "update ke baad bugs aa gaye, bekar"],
     ("Positive", "Build"): ["Design looks premium, nice build", "body bohat achi hai, looks great"],
     ("Neutral", "Q"): ["Which model should I buy under 40k?", "kya ye phone lena chahiye?", "Anyone using this for 6 months? Please share review", "price kya hai Lahore mein?"],
    }
    keys = list(T)
    probs = {"YouTube": {"Positive": .45, "Neutral": .25, "Negative": .30},
             "Facebook Group": {"Positive": .30, "Neutral": .20, "Negative": .50}}
    rows = []
    for _ in range(n):
        plat = rng.choice(["YouTube", "Facebook Group"])
        sent = rng.choices(SENT_ORDER, weights=[probs[plat][s] for s in SENT_ORDER])[0]
        key = rng.choice([k for k in keys if k[0] == sent])
        text = rng.choice(T[key]) + rng.choice(["", " yaar", " honestly", " tbh", " bhai", " overall", " guys", " lol", ". Just my view", " (6 months use)"])
        ts = pd.Timestamp("2026-06-01") + pd.Timedelta(days=rng.randint(0, 135), hours=rng.randint(8, 23), minutes=rng.randint(0, 59))
        rows.append({"platform": plat,
                     "source": f"DEMO {'video' if plat == 'YouTube' else 'group post'} {rng.randint(1, 4)}",
                     "post_type": rng.choice(["review_video", "unboxing_video"]) if plat == "YouTube" else rng.choice(["experience_review", "buying_question", "sale_listing"]),
                     "date": ts.strftime("%Y-%m-%d %H:%M"), "text": text,
                     "likes": int(rng.lognormvariate(1.2 if sent != "Negative" else 1.6, 1.0)), "reply_count": rng.choice([0, 0, 0, 1, 2, 5])})
    for t in ["Subscribe to my channel for free giveaway!", "WhatsApp 03001234567 for cheap phones", "follow me for earn money online"]:
        rows += [{"platform": "YouTube", "source": "DEMO video 1", "post_type": "review_video", "date": "2026-07-01 12:00",
                  "text": t, "likes": 0, "reply_count": 0}] * 2
    rows += rows[:6]  # a few exact duplicates
    return pd.DataFrame(rows)

if SOURCE == "youtube":
    raw = fetch_youtube(YOUTUBE_VIDEOS, YOUTUBE_API_KEY, MAX_COMMENTS_PER_VIDEO, YOUTUBE_ORDER)
elif SOURCE == "paste":
    raw = load_paste(PASTE_BATCHES)
elif SOURCE == "csv":
    raw = load_csv(CSV_PATH)
elif SOURCE == "demo":
    raw = make_demo()
    print("⚠️ DEMO MODE: these are fake comments for testing only.")
else:
    raise ValueError("SOURCE must be youtube / paste / csv / demo")

if raw.empty:
    raise ValueError("No comments loaded. Check your links / paste batches / CSV.")
print(f"Loaded {len(raw)} raw comments from {raw['platform'].nunique()} platform(s)")
raw.head()

In [ ]:
# ===================== 3. CLEAN + FEATURES =====================
SPAM_RE = re.compile(r"whatsapp|wa\.me|telegram|subscribe|follow me|check my|dm me|giveaway|earn money|\b03\d{2}[- ]?\d{7}\b", re.I)
URL_RE = re.compile(r"https?://\S+|www\.\S+")
URDU_RE = re.compile(r"[\u0600-\u06FF]")
RU_MARKERS = set("""hai hain nahi nahin nhi bohat bahut bhai acha accha achha achi kya kyun kyu mein aur yeh woh tha thi hota hoti
karo karta karti lo lein mat bilkul zabardast bekar kharab paisa paisay paise sasta mehnga mehenga lena liya liye
ke ki ka ko se par pe bhi sirf abhi bas lagta lagti chalta chalti wala walay waly""".split())
POS_EMOJI, NEG_EMOJI = set("😍👍❤🔥👌😊😁🥰💯😎🤩✅"), set("😡👎😠🤮😤💩😞😢🤬😭😒❌")

def parse_ts(x):
    if pd.isna(x) or str(x).strip() == "":
        return pd.NaT, False
    s = str(x).strip()
    try:
        t = pd.Timestamp(s)
    except Exception:
        return pd.NaT, False
    if t.tzinfo is not None:
        t = t.tz_convert("Asia/Karachi").tz_localize(None)   # YouTube gives UTC -> Pakistan time
    return t, (":" in s)

def detect_language(t):
    letters = [c for c in t if c.isalpha()]
    if not letters:
        return "Other"
    if len(URDU_RE.findall(t)) / len(letters) > 0.3:
        return "Urdu script"
    toks = re.findall(r"[a-z']+", t.lower())
    hits = sum(w in RU_MARKERS for w in toks)
    return "Roman Urdu" if hits >= 2 or (toks and hits / len(toks) >= 0.25) else "English"

df = raw.copy()
n0 = len(df)
df["text"] = df["text"].astype(str).str.replace(URL_RE, " ", regex=True).str.replace(r"\s+", " ", regex=True).str.strip()
df = df[df["text"].str.len() > 1]
spam_mask = df["text"].str.contains(SPAM_RE)
n_spam = int(spam_mask.sum()); df = df[~spam_mask]
has_signal = df["text"].apply(lambda s: any(c.isalnum() for c in s) or any(c in POS_EMOJI | NEG_EMOJI for c in s))
df = df[has_signal]
df["_key"] = (df["platform"].astype(str) + df["source"].astype(str) + df["text"].str.lower()).str.replace(r"\W+", "", regex=True)
n_before = len(df); df = df.drop_duplicates("_key").drop(columns="_key"); n_dupes = n_before - len(df)

parsed = df["date"].apply(parse_ts)
df["ts"] = pd.to_datetime([p[0] for p in parsed]); df["has_time"] = [p[1] for p in parsed]
df["likes"] = pd.to_numeric(df["likes"], errors="coerce").fillna(0).clip(lower=0)
df["reply_count"] = pd.to_numeric(df["reply_count"], errors="coerce").fillna(0).clip(lower=0)
df["engagement"] = df["likes"] + df["reply_count"]
df["log_eng"] = np.log1p(df["engagement"])
df["language"] = df["text"].apply(detect_language)
df["word_count"] = df["text"].str.split().str.len()
df["dow"] = pd.Categorical(df["ts"].dt.day_name().str[:3], categories=DAYS, ordered=True)
df["hour"] = np.where(df["has_time"], df["ts"].dt.hour, np.nan)
df["month"] = df["ts"].dt.to_period("M").astype(str).replace("NaT", np.nan)
df["comment_id"] = (df["platform"].astype(str) + df["source"].astype(str) + df["text"]).apply(lambda s: hashlib.md5(s.encode()).hexdigest()[:10])
if MODEL_KEYWORDS:
    pat = "|".join(re.escape(k) for k in MODEL_KEYWORDS)
    df["model"] = df["text"].str.extract(f"({pat})", flags=re.I, expand=False).str.lower()
df = df.reset_index(drop=True)
print(f"Raw {n0} -> clean {len(df)}  (spam removed: {n_spam}, duplicates removed: {n_dupes})")
print(df["language"].value_counts().to_string())

In [ ]:
# ===================== 4. LABEL: sentiment + aspects + trust flag =====================
POS = {w: 1 for w in """good great nice smooth fast worth satisfied happy recommend recommended solid reliable impressed decent beautiful
love loved acha accha achha achi achay sahi pasand mazboot tez umda اچھا اچھی پسند عمدہ""".split()}
POS.update({w: 2 for w in """excellent best amazing awesome perfect superb fantastic zabardast kamal shandar behtreen behtareen lajawab
بہترین زبردست شاندار کمال""".split()})
NEG = {w: -1 for w in """bad poor disappointed disappointing heating heats lag slow hang hangs hanging crash crashes problem problems issue issues
refund drain drains dead expensive overpriced zaya mehnga mehenga masla masail dikkat takleef garam برا مسئلہ""".split()}
NEG.update({w: -2 for w in """worst terrible awful hate waste scam fraud overheating overheat laggy defect defective broken useless regret horrible
junk trash bakwas bekar kharab ghatiya faltu dhoka dhokha nakli بیکار خراب گھٹیا فضول دھوکہ بکواس""".split()})
POS_PHR = ["paisa wasool", "paisa vasool", "paise wasool", "highly recommend", "value for money", "worth it", "best phone", "must buy"]
NEG_PHR = ["dont buy", "don't buy", "do not buy", "mat lo", "mat lein", "na lein", "mat kharido", "time waste", "paisa waste",
           "paisay zaya", "paise zaya", "never again", "stay away", "avoid this"]
NEGATORS = {"not", "no", "never", "dont", "don't", "didnt", "isnt", "isn't", "wasnt", "cant", "nahi", "nahin", "nhi", "nai", "na", "mat"}
AFTER_NEG = {"nahi", "nahin", "nhi", "nai"}   # Roman Urdu puts the negator after: "acha nahi hai"

ASPECT_RE = {
 "Battery": r"battery|\bbatt\b|charging|charger|\bcharge|\bmah\b|backup|بیٹری",
 "Camera": r"camera|\bcam\b|photo|picture|\bpics?\b|selfie|\blens|megapixel|\bmp\b|کیمرہ",
 "Heating": r"\bheat|overheat|garam|گرم",
 "Performance": r"\blag|laggy|\bslow|\bhang|\bfast\b|smooth|processor|chipset|\bram\b|gaming|pubg|performance|speed|\bcrash|helio|snapdragon|dimensity",
 "Display": r"display|screen|amoled|\blcd\b|brightness|refresh rate|\btouch|ڈسپلے",
 "Price/Value": r"price|\bcost|expensive|overpriced|\bcheap|affordable|budget|paisa|paise|paisay|qeemat|kimat|mehn?ga|mehenga|sasta|\brs\.? ?\d|\bpkr\b|قیمت|مہنگا|سستا",
 "Software": r"software|update|android|\bui\b|\bads\b|bloatware|\bos\b|\bbugs?\b|hios|xos|miui",
 "Build/Design": r"\bbuild\b|\bbody\b|plastic|design|\bglass|\bweight|\blooks?\b|durab|colou?r",
 "After-sales/Warranty": r"warranty|guarantee|service cent|customer care|customer service|\bsupport\b|repair|replace|\bclaim|after[- ]sales",
}
TRUST_RE = re.compile(
 r"\bfake\b|duplicate|refurb|repack|\bscam|fraud|dhok|nakli|not original|original nahi|asli nahi|non[- ]?pta|not pta|pta nahi|"
 r"\bnumber ?2\b|never again|"
 r"(?:don'?t|dont|do not|\bna\b|mat|nahi|nhi)\W+(?:\w+\W+){0,2}(?:trust|bharosa|bharosay|aitbar|aitebar)|"
 r"(?:trust|bharosa|bharosay|aitbar)\W+(?:\w+\W+){0,2}(?:nahi|nhi|not|nai)", re.I)
ASPECT_COMPILED = {k: re.compile(v, re.I) for k, v in ASPECT_RE.items()}

def rule_label(text):
    t = re.sub(r"(.)\1{2,}", r"\1\1", text.lower())
    score = 0.0
    # negation only counts inside the same clause ("bharosa nahi raha, scam" must not flip "scam")
    for clause in re.split(r"[,.;!?\n]+|\bbut\b|\blekin\b|\bmagar\b|\bhowever\b", t):
        toks = re.findall(r"[\w']+", clause)
        for i, w in enumerate(toks):
            wt = POS.get(w, 0) or NEG.get(w, 0)
            if not wt:
                continue
            before = any(toks[j] in NEGATORS for j in range(max(0, i - 2), i))
            after = any(toks[j] in AFTER_NEG for j in range(i + 1, min(len(toks), i + 3)))
            score += -wt if (before or after) else wt
    score += sum(2 for p in POS_PHR if p in t) + sum(-2 for p in NEG_PHR if p in t)
    score += min(2, sum(c in POS_EMOJI for c in text)) - min(2, sum(c in NEG_EMOJI for c in text))
    t_trust = re.sub(r"\b(?:not|no|never|nahi|isn't|isnt)\s+(?:a\s+|an\s+)?(?:fake|duplicate|refurbished|scam|fraud)", " ", t)
    trust = bool(TRUST_RE.search(t_trust))
    if trust:
        score -= 2          # a trust problem (fake / refurbished / scam / distrust) is a negative opinion
    sent = "Positive" if score >= 1 else "Negative" if score <= -1 else "Neutral"
    aspects = [a for a, rx in ASPECT_COMPILED.items() if rx.search(t)]
    if trust:
        aspects.append("Authenticity/Trust")
    return sent, score, aspects, trust

def parse_llm_json(txt):
    txt = re.sub(r"```(?:json)?", "", txt).strip()
    a, b = txt.find("["), txt.rfind("]")
    return json.loads(txt[a:b + 1])

def label_with_claude(texts, api_key, model, brand, batch=25):
    """Returns list of (sentiment, aspects, trust) or None per text (None -> fall back to rules)."""
    if not api_key:
        raise ValueError("Set ANTHROPIC_API_KEY in cell 1 or use LABEL_MODE='rules'.")
    out = [None] * len(texts)
    system = (f"You label Pakistani social-media comments (English, Urdu, Roman Urdu) about the smartphone brand {brand}. "
              f"Return ONLY a JSON array, no prose. Allowed aspects: {ASPECT_NAMES}.")
    for s in range(0, len(texts), batch):
        chunk = texts[s:s + batch]
        user = ("For each numbered comment return {\"i\": index, \"sentiment\": \"Positive|Neutral|Negative\", "
                "\"aspects\": [allowed aspects the comment discusses], \"trust_issue\": true/false (fake/refurbished/non-PTA/scam/distrust of brand)}.\n\n"
                + "\n".join(f"{i}: {t[:400]}" for i, t in enumerate(chunk)))
        for attempt in range(3):
            try:
                r = requests.post("https://api.anthropic.com/v1/messages",
                                  headers={"x-api-key": api_key, "anthropic-version": "2023-06-01", "content-type": "application/json"},
                                  json={"model": model, "max_tokens": 3000, "system": system,
                                        "messages": [{"role": "user", "content": user}]}, timeout=90)
                r.raise_for_status()
                for item in parse_llm_json(r.json()["content"][0]["text"]):
                    i = int(item["i"])
                    if 0 <= i < len(chunk) and item.get("sentiment") in SENT_ORDER:
                        out[s + i] = (item["sentiment"], [a for a in item.get("aspects", []) if a in ASPECT_NAMES], bool(item.get("trust_issue", False)))
                break
            except Exception as e:
                print(f"  batch {s // batch + 1} attempt {attempt + 1} failed: {str(e)[:100]}")
                time.sleep(2 * (attempt + 1))
    return out

res = df["text"].apply(rule_label)
df["sentiment"] = [r[0] for r in res]; df["score"] = [r[1] for r in res]
df["aspects"] = [r[2] for r in res]; df["trust_flag"] = [r[3] for r in res]
df["label_source"] = "rules"

if LABEL_MODE == "claude":
    print("Labeling with Claude...")
    llm = label_with_claude(df["text"].tolist(), ANTHROPIC_API_KEY, LLM_MODEL, BRAND)
    for i, r in enumerate(llm):
        if r is not None:
            df.at[i, "sentiment"], df.at[i, "aspects"], df.at[i, "trust_flag"] = r[0], r[1], r[2]
            df.at[i, "label_source"] = "claude"
    print(df["label_source"].value_counts().to_string())

df["sentiment"] = pd.Categorical(df["sentiment"], categories=SENT_ORDER, ordered=True)
print(df["sentiment"].value_counts().reindex(SENT_ORDER).to_string())

In [ ]:
# ===================== 5. VALIDATE (hand-label 150 comments once) =====================
# Step A: run this cell -> downloads gold_to_label.csv. Fill the `human_sentiment` column (Positive/Neutral/Negative).
# Step B: upload as gold_labeled.csv, set GOLD_LABELED_PATH = "gold_labeled.csv" in cell 1, and re-run this cell.
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, classification_report, confusion_matrix

gold = df.sample(min(150, len(df)), random_state=42)[["comment_id", "platform", "text"]].assign(human_sentiment="")
gold.to_csv(f"{OUT_DIR}/gold_to_label.csv", index=False)
VALID = {}
if GOLD_LABELED_PATH and os.path.exists(GOLD_LABELED_PATH):
    g = pd.read_csv(GOLD_LABELED_PATH)
    g["human_sentiment"] = g["human_sentiment"].astype(str).str.strip().str.title()
    g = g[g["human_sentiment"].isin(SENT_ORDER)].merge(df[["comment_id", "sentiment"]], on="comment_id", how="inner")
    if len(g) >= 20:
        y, p = g["human_sentiment"], g["sentiment"].astype(str)
        VALID = {"n": len(g), "accuracy": accuracy_score(y, p), "macro_f1": f1_score(y, p, average="macro", labels=SENT_ORDER),
                 "kappa": cohen_kappa_score(y, p, labels=SENT_ORDER)}
        print(f"Validated on {VALID['n']} hand-labeled comments")
        print(f"Accuracy {VALID['accuracy']:.2f} | Macro-F1 {VALID['macro_f1']:.2f} | Cohen's kappa {VALID['kappa']:.2f}")
        print(classification_report(y, p, labels=SENT_ORDER, zero_division=0))
        cm = pd.DataFrame(confusion_matrix(y, p, labels=SENT_ORDER), index=[f"true {s}" for s in SENT_ORDER], columns=[f"pred {s}" for s in SENT_ORDER])
        plt.figure(figsize=(6, 4)); sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
        plt.title("Sentiment labeling vs hand labels" + TAG); plt.tight_layout()
        plt.savefig(f"{OUT_DIR}/charts/00_validation_confusion.png", dpi=150); plt.show()
    else:
        print("Need at least 20 valid hand labels (Positive/Neutral/Negative).")
else:
    print(f"Saved {OUT_DIR}/gold_to_label.csv ({len(gold)} comments). Label it, then follow Step B above.")
    if IN_COLAB:
        from google.colab import files
        files.download(f"{OUT_DIR}/gold_to_label.csv")

In [ ]:
# ===================== 6. ANALYSIS: answers to the brief =====================
def net(s):
    return (((s == "Positive").sum() - (s == "Negative").sum()) / max(len(s), 1)) * 100

def done(name):
    plt.tight_layout(); plt.savefig(f"{OUT_DIR}/charts/{name}.png", dpi=150, bbox_inches="tight"); plt.show()

def safe(fn):
    try:
        fn()
    except Exception as e:
        plt.close("all"); print(f"  (chart skipped: {fn.__name__}: {e})")

# Q: Which platform shows more positive / negative opinion?
def c1():
    ct = pd.crosstab(df.platform, df.sentiment, normalize="index").reindex(columns=SENT_ORDER, fill_value=0) * 100
    ax = ct.plot(kind="barh", stacked=True, color=[COLORS[s] for s in SENT_ORDER], figsize=(9, 1.5 + 0.9 * len(ct)))
    ax.set_xlabel("% of comments"); ax.set_title("1. Sentiment by platform" + TAG); done("01_sentiment_by_platform")
def c2():
    t = df.groupby("platform").apply(lambda g: pd.Series({"net": net(g.sentiment), "n": len(g)})).sort_values("net")
    ax = t["net"].plot(kind="barh", color=["#e53935" if v < 0 else "#43a047" for v in t["net"]], figsize=(9, 1.5 + 0.8 * len(t)))
    for i, (v, n) in enumerate(zip(t["net"], t["n"])): ax.text(v, i, f"  n={int(n)}", va="center")
    ax.axvline(0, color="k", lw=.8); ax.set_title("2. Net sentiment score by platform (positive% − negative%)" + TAG); done("02_net_sentiment_platform")

# Q: Which topics/aspects does the audience talk about, and which cause complaints?
ex = df.explode("aspects").dropna(subset=["aspects"]).reset_index(drop=True)
def c3():
    ct = pd.crosstab(ex.aspects, ex.sentiment).reindex(columns=SENT_ORDER, fill_value=0)
    ct = ct.loc[ct.sum(axis=1).sort_values().index]
    ct.plot(kind="barh", stacked=True, color=[COLORS[s] for s in SENT_ORDER], figsize=(10, 5))
    plt.xlabel("comments mentioning aspect"); plt.title("3. What people talk about, and how they feel about it" + TAG); done("03_aspects_by_sentiment")
def c4():
    t = ex.groupby("aspects").apply(lambda g: (g.sentiment == "Negative").mean() * 100)
    counts = ex.aspects.value_counts().reindex(t.index)
    k = 5 if (counts >= 5).any() else 2          # small samples: lower the minimum
    t = t[counts >= k].sort_values(); assert len(t), "no aspect has enough mentions"
    ax = t.plot(kind="barh", color="#e53935", figsize=(9, 4)); ax.set_xlabel("% of mentions that are negative")
    ax.set_title(f"4. Complaint rate by aspect (min {k} mentions)" + TAG); done("04_complaint_rate_by_aspect")

# Q: Does sentiment drive engagement? Which post type / language engages more?
def c5():
    data = [df[df.sentiment == s].log_eng.values for s in SENT_ORDER]
    fig, ax = plt.subplots(figsize=(8, 5)); bp = ax.boxplot(data, patch_artist=True)
    for patch, s in zip(bp["boxes"], SENT_ORDER): patch.set_facecolor(COLORS[s])
    ax.set_xticklabels(SENT_ORDER); ax.set_ylabel("log(1 + likes + replies)")
    ax.set_title("5. Engagement by comment sentiment" + TAG); done("05_engagement_by_sentiment")
def c6():
    t = df.groupby("post_type").engagement.agg(["mean", "median", "count"]); t = t[t["count"] >= 3].sort_values("mean")
    t[["mean", "median"]].plot(kind="barh", figsize=(9, 1.5 + 0.8 * len(t))); plt.xlabel("likes + replies per comment")
    plt.title("6. Engagement by post type" + TAG); done("06_engagement_by_post_type")

# Q: When is the audience active?
def c7():
    d = df.dropna(subset=["hour"])
    if len(d) >= 0.5 * len(df):
        pv = d.pivot_table(index="dow", columns="hour", values="comment_id", aggfunc="count", fill_value=0).reindex(DAYS).reindex(columns=range(24), fill_value=0)
        plt.figure(figsize=(12, 4)); sns.heatmap(pv, cmap="YlOrRd", cbar_kws={"label": "comments"}); plt.xlabel("hour (Pakistan time)")
        plt.title("7. When comments are posted (day × hour)" + TAG)
    else:
        df.dow.value_counts().reindex(DAYS).plot(kind="bar", figsize=(8, 4)); plt.title("7. Comments by day of week" + TAG)
    done("07_when_posted")
def c8():
    m = df.dropna(subset=["month"]).groupby("month").apply(lambda g: pd.Series({"net": net(g.sentiment), "n": len(g)}))
    m = m[m.n >= 5]; assert len(m) >= 3, "need 3+ months with 5+ comments"
    ax = m["net"].plot(marker="o", figsize=(9, 4)); ax.axhline(0, color="k", lw=.8); ax.set_ylabel("net sentiment")
    ax.set_title("8. Net sentiment over time" + TAG); done("08_sentiment_trend")

# Q: Which audience segment (language) is more responsive / negative?
def c9():
    t = df.groupby("language").apply(lambda g: pd.Series({"net": net(g.sentiment), "median_eng": g.engagement.median(), "n": len(g)}))
    t = t[t.n >= 5]
    fig, ax = plt.subplots(1, 2, figsize=(11, 4)); t["net"].plot(kind="bar", ax=ax[0], color="#5c6bc0"); ax[0].set_title("Net sentiment")
    t["median_eng"].plot(kind="bar", ax=ax[1], color="#ffa726"); ax[1].set_title("Median engagement")
    fig.suptitle("9. Audience segment by comment language" + TAG, fontweight="bold"); done("09_language_segments")

# Q: Brand image: how much distrust / authenticity talk is there?
def c10():
    t = df.groupby("platform").trust_flag.mean() * 100
    ax = t.plot(kind="bar", color="#8e24aa", figsize=(7, 4)); ax.set_ylabel("% of comments"); plt.xticks(rotation=0)
    ax.set_title("10. Trust / authenticity complaints by platform" + TAG); done("10_trust_flag_rate")

for f in [c1, c2, c3, c4, c5, c6, c7, c8, c9, c10]:
    safe(f)

# Statistical checks (report these in one line each)
TESTS = {}
if df.platform.nunique() > 1:
    chi2, p, _, _ = stats.chi2_contingency(pd.crosstab(df.platform, df.sentiment)); TESTS["platform_vs_sentiment_p"] = p
    print(f"Sentiment differs by platform? chi-square p = {p:.4f}")
groups = [g.log_eng.values for _, g in df.groupby("sentiment") if len(g) >= 5]
if len(groups) >= 2:
    H, p = stats.kruskal(*groups); TESTS["engagement_vs_sentiment_p"] = p
    print(f"Engagement differs by sentiment? Kruskal-Wallis p = {p:.4f}")

# Most common words in negative comments
STOP = set("""the a an is it i to of and in for on this that was with are be at as but have has my me you your so not no very too
hai hain ka ki ke ko se me mein ne ye yeh woh wo to bhi aur ya ho tha thi hota hoti kar karo kya par pe se phone mobile set yaar bhai just really""".split()) | set(re.findall(r"\w+", BRAND.lower()))
neg_words = Counter(w for t in df[df.sentiment == "Negative"].text for w in re.findall(r"[\w']+", t.lower()) if len(w) > 2 and w not in STOP)
TOP_NEG = neg_words.most_common(12)
print("Top words in negative comments:", TOP_NEG)

In [ ]:
# ===================== 7. SUMMARY TABLES + AI-ASSISTED RECOMMENDATION =====================
def agg_tbl(by, data):
    return data.groupby(by).apply(lambda g: pd.Series({
        "n": len(g), "positive_%": (g.sentiment == "Positive").mean() * 100, "negative_%": (g.sentiment == "Negative").mean() * 100,
        "net_sentiment": net(g.sentiment), "median_engagement": g.engagement.median(), "mean_engagement": g.engagement.mean(),
        "trust_flag_%": g.trust_flag.mean() * 100})).round(1).astype({"n": int})

TABLES = {"platform": agg_tbl("platform", df), "language": agg_tbl("language", df), "post_type": agg_tbl("post_type", df),
          "day_of_week": agg_tbl("dow", df), "sentiment_engagement": df.groupby("sentiment").engagement.agg(["count", "mean", "median"]).round(2)}
TABLES["aspect"] = ex.groupby("aspects").apply(lambda g: pd.Series({
    "mentions": len(g), "negative_%": (g.sentiment == "Negative").mean() * 100, "positive_%": (g.sentiment == "Positive").mean() * 100,
    "median_engagement": g.engagement.median()})).round(1).astype({"mentions": int}).sort_values("mentions", ascending=False)
for k, v in TABLES.items():
    print(f"\n--- {k} ---"); print(v.to_string())

asp = TABLES["aspect"]; asp5 = asp[asp.mentions >= 5] if (asp.mentions >= 5).any() else asp[asp.mentions >= 2]
top_complaint = asp5["negative_%"].idxmax() if len(asp5) else "n/a"
top_praise = asp5["positive_%"].idxmax() if len(asp5) else "n/a"
plat = TABLES["platform"]
best_plat, worst_plat = plat["net_sentiment"].idxmax(), plat["net_sentiment"].idxmin()
busiest_day = df.dow.value_counts().idxmax()
overall_net = net(df.sentiment); trust_pct = df.trust_flag.mean() * 100
med = df.groupby("sentiment").engagement.median()

SUMMARY = f"""Brand: {BRAND} (Pakistan). Comments analysed: {len(df)} from {df.platform.nunique()} platform(s). Labeling: {LABEL_MODE}.
Overall: {(df.sentiment=='Positive').mean()*100:.0f}% positive, {(df.sentiment=='Neutral').mean()*100:.0f}% neutral, {(df.sentiment=='Negative').mean()*100:.0f}% negative (net {overall_net:+.0f}).
Net sentiment by platform: {', '.join(f'{k} {v:+.0f}' for k, v in plat['net_sentiment'].items())}.
Most-complained aspect: {top_complaint}; most-praised aspect: {top_praise}.
Aspect table (mentions, % negative): {', '.join(f'{k} ({r.mentions}, {r["negative_%"]:.0f}%)' for k, r in asp.head(8).iterrows())}.
Trust/authenticity complaints (fake, refurbished, non-PTA, scam, distrust): {trust_pct:.1f}% of comments.
Median engagement (likes+replies) by sentiment: {', '.join(f'{k} {v:.1f}' for k, v in med.items())}.
Language mix: {', '.join(f'{k} {v:.0f}%' for k, v in (df.language.value_counts(normalize=True)*100).items())}.
Busiest day: {busiest_day}. Top negative words: {', '.join(w for w, _ in TOP_NEG[:8])}."""
PROMPT = f"""You are a junior digital-marketing analyst advising the Pakistani marketing team of {BRAND} phones.
Use ONLY the data below. Do not invent numbers.

{SUMMARY}

Give: (1) the 3 most important findings, (2) 3 concrete social-media posts to publish next (content type, topic, language mix, best day), (3) 2 customer problems the brand should publicly address and how, (4) 3 caveats about this data."""
print("\n" + "=" * 70 + "\nSUMMARY FOR REPORT\n" + "=" * 70 + "\n" + SUMMARY)

AI_RECO = ""
if ANTHROPIC_API_KEY:
    try:
        r = requests.post("https://api.anthropic.com/v1/messages",
                          headers={"x-api-key": ANTHROPIC_API_KEY, "anthropic-version": "2023-06-01", "content-type": "application/json"},
                          json={"model": LLM_MODEL, "max_tokens": 1500, "messages": [{"role": "user", "content": PROMPT}]}, timeout=90)
        r.raise_for_status(); AI_RECO = r.json()["content"][0]["text"]
        print("\n" + "=" * 70 + "\nAI-ASSISTED RECOMMENDATION\n" + "=" * 70 + "\n" + AI_RECO)
    except Exception as e:
        print("API call failed:", str(e)[:200])
if not AI_RECO:
    print("\n" + "=" * 70 + "\nCOPY THIS PROMPT INTO CLAUDE / ChatGPT:\n" + "=" * 70 + "\n" + PROMPT)
print("\nThen CRITIQUE the AI answer: which claims does your data support, which does it not? (That critique is worth marks.)")

In [ ]:
# ===================== 8. EXPORT: data, tables, charts, report draft =====================
export = df.copy(); export["aspects"] = export["aspects"].apply(lambda a: "; ".join(a))
export.drop(columns=["has_time"]).to_csv(f"{OUT_DIR}/labeled_comments.csv", index=False)
with pd.ExcelWriter(f"{OUT_DIR}/summary_tables.xlsx") as xw:
    for k, v in TABLES.items():
        v.to_excel(xw, sheet_name=k[:30])

demo_warn = "> ⚠️ **DEMO DATA. Replace with real results before submitting.**\n\n" if SOURCE == "demo" else ""
val_line = (f"Sentiment labels were validated against {VALID['n']} hand-labeled comments: accuracy {VALID['accuracy']:.0%}, macro-F1 {VALID['macro_f1']:.2f}, Cohen's kappa {VALID['kappa']:.2f}."
            if VALID else "Sentiment labels were NOT yet validated against hand labels (run cell 5 step B).")
report = f"""# Pakistani Social Media Content Detective: {BRAND} perception study

{demo_warn}## 1. Business question
What do Pakistani social-media users say about {BRAND} phones, which problems drive negative opinion, how much do they distrust the brand, and what should {BRAND} post next?

## 2. Data
{len(df)} public comments after cleaning (raw {n0}; removed {n_spam} spam and {n_dupes} duplicates) from: {', '.join(sorted(df.platform.unique()))}.
Languages: {', '.join(f'{k} {v:.0f}%' for k, v in (df.language.value_counts(normalize=True)*100).items())}.
Usernames were not stored. Only public posts were used.

## 3. Method
Cleaning, language detection, then sentiment / aspect / trust labeling using {'a Pakistan-aware English + Roman Urdu + Urdu lexicon with negation handling' if LABEL_MODE=='rules' else 'Claude with a fixed label schema'}. {val_line}
Engagement = likes + replies per comment. Time converted to Pakistan time.

## 4. Findings
- Overall sentiment: {(df.sentiment=='Positive').mean()*100:.0f}% positive / {(df.sentiment=='Negative').mean()*100:.0f}% negative (net {overall_net:+.0f}).
- Platform: {best_plat} is the most positive (net {plat.loc[best_plat,'net_sentiment']:+.0f}); {worst_plat} is the most negative (net {plat.loc[worst_plat,'net_sentiment']:+.0f}).{' Difference test p=%.3f.' % TESTS['platform_vs_sentiment_p'] if 'platform_vs_sentiment_p' in TESTS else ''}
- Biggest complaint area: **{top_complaint}**; most praised: **{top_praise}**.
- Trust/authenticity complaints appear in {trust_pct:.1f}% of comments.
- Engagement: median likes+replies is {', '.join(f'{k} {v:.1f}' for k, v in med.items())} for Positive/Neutral/Negative comments.{' Kruskal-Wallis p=%.3f.' % TESTS['engagement_vs_sentiment_p'] if 'engagement_vs_sentiment_p' in TESTS else ''}
- Busiest day for comments: {busiest_day}.
(Insert charts from `charts/` here and interpret each in 1–2 sentences.)

## 5. AI-assisted marketing recommendation
{AI_RECO or '(Paste the AI output here, then add your critique of what the data does and does not support.)'}

## 6. Limitations
- People with problems are more likely to comment, so negativity may be overstated.
- Paid or fake comments, sarcasm and mixed Roman Urdu spelling can mislead labeling.
- Aspect sentiment is comment-level: a comment praising the camera but cursing the battery counts once per aspect with one overall label.
- Sample is not representative of all buyers; no causal claims.
"""
open(f"{OUT_DIR}/report_draft.md", "w", encoding="utf-8").write(report)

import shutil
shutil.make_archive("results", "zip", OUT_DIR)
print("Saved results.zip with:", sorted(os.listdir(OUT_DIR)))
if IN_COLAB:
    from google.colab import files
    files.download("results.zip")